# Fully Localized, Serverless GraphRAG with NetworkX & Ollama (Llama 3.1)

In Notebook 1, we mastered graph creation, node/edge attributes, and visualization (`DiGraph`). In Notebook 2, we explored graph traversal, shortest paths, connected components, and centrality metrics (`PageRank`, `Degree Centrality`).

Now, we combine these skills to build a Fully Localized, Serverless GraphRAG (Graph-based Retrieval-Augmented Generation) System from scratch using a factually accurate, real-world AI and computational drug discovery corpus.

### Why Serverless GraphRAG with NetworkX + Ollama?
* Standard Vector RAG chunks text and retrieves isolated paragraphs based on cosine similarity. It often fails when answering a question requires connecting facts across multiple distant paragraphs. GraphRAG explicitly maps relationships `(Subject -> Relation -> Object)` so we can traverse multi-hop paths across documents.
* By pairing NetworkX (in-memory pure Python graphs) with Ollama (`llama3.1`), our entire pipeline runs locally on your machine with zero server overhead and complete data privacy.

In [ ]:
# 1. SETUP & LOCAL ENVIRONMENT CONFIGURATION
import os
import json
import re
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

# Set Matplotlib defaults for crisp visualizations
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["figure.dpi"] = 100

MODEL_NAME = "llama3.2:3b"
INDEX_FILE = "local_graphrag_index.json"

def call_ollama(prompt, system_prompt="You are a helpful AI assistant.", json_mode=False):

    llm = ChatOllama(
        model=MODEL_NAME,
        temperature=0.0,
        format="json" if json_mode else None
    )
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=prompt)
    ]
    response = llm.invoke(messages)
    return response.content.strip()

### Serverless Persistence (Zero-DB JSON Storage)
Because NetworkX lives entirely in-memory, we don't want to re-run LLM extraction every time we restart our notebook. To avoid this, we use `nx.node_link_data` to serialize our knowledge graph including node categories, edge relations, and source evidence into a single portable local JSON file (`local_graphrag_index.json`).

In [ ]:
def save_graphrag_index(G, filepath=INDEX_FILE):
    """Serializes the NetworkX graph to a local JSON file."""
    data = nx.node_link_data(G)
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)
    print(f"Saved GraphRAG index ({G.number_of_nodes()} nodes, {G.number_of_edges()} edges) to '{filepath}'.")

def load_graphrag_index(filepath=INDEX_FILE):
    """Loads a serialized NetworkX graph from a local JSON file."""
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    G_loaded = nx.node_link_graph(data, directed=True)
    print(f"Loaded GraphRAG index ({G_loaded.number_of_nodes()} nodes, {G_loaded.number_of_edges()} edges) from '{filepath}'.")
    return G_loaded

print("Setup complete! Ready to build our local GraphRAG pipeline.")

## 2. Unstructured Corpus & Automated Knowledge Graph Extraction

To see why GraphRAG excels over standard Vector RAG, let's look at a 4-chunk factual corpus where real-world facts are distributed across separate documents:
* **Chunk 1** introduces a founder (`Demis Hassabis`), his AI research lab (`DeepMind`), and its protein structure prediction model (`AlphaFold 2`).
* **Chunk 2** describes the co-inventor (`John Jumper`) behind the core architecture (`Evoformer`) and his team's (`Protein Structure Team`) computational drug discovery partnership.
* **Chunk 3** explains how that digital biology company (`Isomorphic Labs`) uses `AlphaFold 2` to advance a small-molecule oncology program (`Eli Lilly Oncology Targets`) addressing `Solid Tumors`.
* **Chunk 4** details corporate parent funding (`Alphabet Inc`) and large-scale training compute infrastructure (`Google Cloud TPUv3 Supercluster`).

If a user asks: *"How is Demis Hassabis connected to Eli Lilly Oncology Targets?"*, no single text chunk contains both names!

Let's use `llama3.1` to extract structured triplets and build a unified NetworkX `DiGraph`.

In [ ]:
# 2. UNSTRUCTURED CORPUS & LLM-DRIVEN TRIPLET EXTRACTION

corpus = {
    "Chunk 1": (
        "Dr. Demis Hassabis, a former researcher at University College London, founded DeepMind in 2010. "
        "DeepMind develops AlphaFold 2, a next-generation protein structure prediction model. "
        "AlphaFold 2 uses the Evoformer architecture to process multiple sequence alignments efficiently."
    ),
    "Chunk 2": (
        "John Jumper co-invented the Evoformer architecture. "
        "John Jumper leads the Protein Structure Team in London. Recently, the Protein Structure Team "
        "partnered with Isomorphic Labs to accelerate computational drug discovery."
    ),
    "Chunk 3": (
        "Isomorphic Labs uses the AlphaFold 2 model to simulate molecular binding pathways. "
        "Through these simulations, Isomorphic Labs advances Eli Lilly Oncology Targets, a novel small-molecule compound. "
        "Eli Lilly Oncology Targets effectively address Solid Tumors in preclinical research."
    ),
    "Chunk 4": (
        "Alphabet Inc funded both DeepMind and Isomorphic Labs during their corporate expansions. "
        "To support large-scale biomedical runs, AlphaFold 2 was trained on the Google Cloud TPUv3 Supercluster."
    )
}

EXTRACTION_SYSTEM_PROMPT = """
You are an expert Knowledge Graph extraction system.
Extract all meaningful entity relationships from the text as a JSON object with a single key "triplets".
Each item in "triplets" must have:
- "subject": Standardized entity name (Title Case, omit titles like Dr. or Prof.)
- "subject_type": Category (e.g., Person, Organization, Model, Architecture, Compound, Condition, Infrastructure)
- "relation": UPPERCASE_VERB_PHRASE (e.g., FOUNDED, DEVELOPS, USES_ARCHITECTURE, CO_INVENTED, ADVANCES)
- "object": Standardized entity name (Title Case, omit titles like Dr. or Prof.)
- "object_type": Category
Return ONLY valid JSON.
"""

def build_knowledge_graph_from_corpus(corpus_dict):
    """Extracts triplets from each text chunk using Llama 3.1 and populates a NetworkX DiGraph."""
    G = nx.DiGraph()
    
    for chunk_id, text in corpus_dict.items():
        print(f"Extracting triplets from {chunk_id}...")
        user_prompt = f"CHUNK: {chunk_id}\nTEXT: {text}"
        raw_json = call_ollama(user_prompt, system_prompt=EXTRACTION_SYSTEM_PROMPT, json_mode=True)
        
        try:
            data = json.loads(raw_json)
            triplets = data.get("triplets", [])
        except json.JSONDecodeError:
            print(f"  [Warning] Failed to parse JSON for {chunk_id}")
            triplets = []
            
        for t in triplets:
            subj = t["subject"].strip()
            obj = t["object"].strip()
            rel = t["relation"].strip()
            
            # Add or update subject node with metadata
            G.add_node(subj, category=t.get("subject_type", "Entity"), source_chunk=chunk_id)
            # Add or update object node with metadata
            G.add_node(obj, category=t.get("object_type", "Entity"), source_chunk=chunk_id)
            
            # Add directed edge with relationship label and source text provenance
            G.add_edge(
                subj, obj,
                relation=rel,
                source_chunk=chunk_id,
                evidence=text
            )
            
    return G

# Build or load the GraphRAG index (First run requires Llama 3.1; subsequent runs load from cache)
if os.path.exists(INDEX_FILE):
    print(f"Cached index found! Loading directly from '{INDEX_FILE}'...")
    kg = load_graphrag_index(INDEX_FILE)
else:
    print("No cache found. Querying Llama 3.1 via LangChain to build the Knowledge Graph...")
    kg = build_knowledge_graph_from_corpus(corpus)
    save_graphrag_index(kg, INDEX_FILE)

print(f"\nKnowledge Graph Ready! Nodes: {kg.number_of_nodes()} | Directed Edges: {kg.number_of_edges()}")

## 3. Auditing & Visualizing the Extracted Knowledge Graph

Before building our retrieval engine, let's apply the topological metrics (`PageRank`, `Degree Centrality`, and `Weakly Connected Components`) to audit our LLM-constructed graph and visualize its structure.

In [ ]:
# 3. TOPOLOGY AUDIT & COLOR-CODED VISUALIZATION

# Compute Centrality Metrics
pagerank_scores = nx.pagerank(kg, alpha=0.85)
degree_centrality = nx.degree_centrality(kg)

# Store metrics directly as node attributes in NetworkX
nx.set_node_attributes(kg, pagerank_scores, "pagerank")
nx.set_node_attributes(kg, degree_centrality, "degree_centrality")

# Table
audit_rows = []
for node, data in kg.nodes(data=True):
    audit_rows.append({
        "Entity": node,
        "Category": data.get("category", "Unknown"),
        "Degree Centrality": round(data.get("degree_centrality", 0), 3),
        "PageRank": round(data.get("pagerank", 0), 4),
        "Total Connections (In+Out)": kg.degree(node)
    })

audit_df = pd.DataFrame(audit_rows).sort_values(by="PageRank", ascending=False).reset_index(drop=True)
print(f"Weakly Connected Components: {nx.number_weakly_connected_components(kg)}")
print(f"Graph Density: {nx.density(kg):.4f}\n")
display(audit_df.head(8))

# Color-Coded Knowledge Graph Visualization
CATEGORY_COLORS = {
    "Person": "#FF6B6B",
    "Organization": "#4ECDC4",
    "Model": "#FFD166",
    "Architecture": "#A06CD5",
    "Compound": "#06D6A0",
    "Condition": "#EF476F",
    "Infrastructure": "#118AB2"
}

def plot_knowledge_graph(G, highlight_nodes=None, highlight_edges=None, title="Extracted Knowledge Graph"):
    plt.figure(figsize=(13, 8))
    pos = nx.spring_layout(G, k=1.1, iterations=60, seed=42)
    
    # Node colors and sizes scaled by degree
    node_colors = [CATEGORY_COLORS.get(G.nodes[n].get("category", ""), "#CCCCCC") for n in G.nodes()]
    node_sizes = [800 + (G.degree(n) * 350) for n in G.nodes()]
    
    # Dim non-highlighted elements if a reasoning path is active
    if highlight_nodes:
        alphas = [1.0 if n in highlight_nodes else 0.25 for n in G.nodes()]
    else:
        alphas = 0.95
        
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=alphas, edgecolors="black", linewidths=1.5)
    nx.draw_networkx_labels(G, pos, font_size=9, font_weight="bold")
    
    # Draw base edges
    edge_colors = []
    edge_widths = []
    for u, v in G.edges():
        if highlight_edges and ((u, v) in highlight_edges or (v, u) in highlight_edges):
            edge_colors.append("#E63946")
            edge_widths.append(3.0)
        else:
            edge_colors.append("#888888" if not highlight_edges else "#DDDDDD")
            edge_widths.append(1.5 if not highlight_edges else 1.0)
            
    nx.draw_networkx_edges(G, pos, edge_color=edge_colors, width=edge_widths, arrows=True, arrowsize=18, connectionstyle="arc3,rad=0.08")
    
    # Edge relationship labels
    edge_labels = {(u, v): d["relation"] for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7.5, font_color="#2B2D42")
    
    # Legend for Entity Categories
    for cat, color in CATEGORY_COLORS.items():
        plt.scatter([], [], c=color, label=cat, s=120, edgecolors="black")
    plt.legend(scatterpoints=1, frameon=True, title="Entity Categories", loc="upper left")
    plt.title(title, fontsize=14, fontweight="bold")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

plot_knowledge_graph(kg, title="Full Extracted Knowledge Graph (Node Size = Degree)")

## 4. Graph Retrieval Engine: Entity Linking & Multi-Hop Subgraph Traversal

When a user asks a question, our Graph Retrieval Engine performs three steps:
1. **Entity Linking:** Maps the natural language question to matching seed nodes in `kg` (using case-insensitive substring matching + LLM entity linking).
2. **Multi-Hop Path & Ego-Graph Retrieval:** 
   * Finds the undirected shortest paths (`nx.shortest_path`) connecting all matched seed entities across the graph so we never miss edges that point in opposite directions.
   * Expands a $k$-hop local neighborhood (`nx.ego_graph`) around each seed entity.
3. **Centrality-Weighted Context Pruning:** Ranks retrieved relationships by the average PageRank/Degree Centrality of their endpoints so `llama3.1` receives the highest-signal facts first.

In [ ]:
# 4. GRAPH RETRIEVAL ENGINE (ENTITY LINKING + MULTI-HOP PATHS)

def extract_query_entities(query, G):
    """Links a user query to existing nodes in the NetworkX graph."""
    graph_nodes = list(G.nodes())
    
    # Step 1: Fast case-insensitive substring match
    matched = [node for node in graph_nodes if node.lower() in query.lower()]
    if len(matched) >= 2:
        return matched
        
    # Step 2: LLM-assisted entity linking fallback for fuzzy/synonym queries
    prompt = f"ENTITIES: {json.dumps(graph_nodes)}\nQUESTION: {query}\nReturn JSON with key 'matched_entities' containing exact node names from ENTITIES relevant to QUESTION."
    raw = call_ollama(prompt, system_prompt="You are a precise entity linker. Output valid JSON only.", json_mode=True)
    try:
        llm_matches = json.loads(raw).get("matched_entities", [])
        for m in llm_matches:
            if m in G.nodes() and m not in matched:
                matched.append(m)
    except Exception:
        pass
    return matched

def retrieve_graph_context(query, G, k_hops=1, top_k_edges=10):
    """
    Retrieves a multi-hop subgraph connecting query entities + k-hop neighborhoods,
    ranked by node centrality.
    """
    seed_nodes = extract_query_entities(query, G)
    print(f"Matched Seed Entities in Graph: {seed_nodes}")
    
    if not seed_nodes:
        return "", set(), set()
        
    undirected_G = G.to_undirected()
    retrieved_nodes = set(seed_nodes)
    path_edges = set()
    
    # 1. Multi-Hop Shortest Paths between all pairs of seed entities
    for i in range(len(seed_nodes)):
        for j in range(i + 1, len(seed_nodes)):
            u, v = seed_nodes[i], seed_nodes[j]
            if nx.has_path(undirected_G, u, v):
                for path in nx.all_shortest_paths(undirected_G, source=u, target=v):
                    retrieved_nodes.update(path)
                    for idx in range(len(path) - 1):
                        n1, n2 = path[idx], path[idx + 1]
                        if G.has_edge(n1, n2):
                            path_edges.add((n1, n2))
                        if G.has_edge(n2, n1):
                            path_edges.add((n2, n1))
                            
    # 2. K-Hop Ego-Graph expansion around seed nodes for local context
    for seed in seed_nodes:
        ego = nx.ego_graph(undirected_G, seed, radius=k_hops)
        retrieved_nodes.update(ego.nodes())
        
    # Induced subgraph containing all discovered nodes
    subgraph = G.subgraph(retrieved_nodes)
    
    # 3. Rank edges: prioritize direct multi-hop bridge paths first, then rank by endpoint PageRank
    scored_edges = []
    for u, v, data in subgraph.edges(data=True):
        is_bridge_path = 1 if (u, v) in path_edges else 0
        importance = G.nodes[u].get("pagerank", 0) + G.nodes[v].get("pagerank", 0)
        scored_edges.append((is_bridge_path, importance, u, v, data))
        
    scored_edges.sort(key=lambda x: (x[0], x[1]), reverse=True)
    selected_edges = scored_edges[:top_k_edges]
    
    # 4. Serialize Subgraph into structured text for the LLM prompt
    triplets_text = []
    evidence_chunks = {}
    active_edges = set()
    active_nodes = set()
    
    for is_bridge, score, u, v, data in selected_edges:
        active_edges.add((u, v))
        active_nodes.update([u, v])
        u_cat = G.nodes[u].get("category", "Entity")
        v_cat = G.nodes[v].get("category", "Entity")
        rel = data["relation"]
        chunk = data["source_chunk"]
        triplets_text.append(f"- ({u} [{u_cat}]) --[{rel}]--> ({v} [{v_cat}])")
        evidence_chunks[chunk] = data["evidence"]
        
    serialized_context = "### RETRIEVED KNOWLEDGE GRAPH PATHWAYS:\n" + "\n".join(triplets_text)
    serialized_context += "\n\n### SOURCE TEXT EVIDENCE:\n"
    for cid, text in sorted(evidence_chunks.items()):
        serialized_context += f"- [{cid}]: {text}\n"
        
    return serialized_context, active_nodes, active_edges

## 5. Context Serialization, Grounded Answer Generation & Visual Trace

Now let's run an end-to-end GraphRAG query that requires hopping across Chunks, inspect the exact context block passed to `llama3.1`, and highlight the retrieved reasoning path directly on our NetworkX graph.

In [ ]:
# 5. END-TO-END GRAPHRAG QUERY & VISUAL REASONING TRACE


RAG_SYSTEM_PROMPT = """
You are a precise GraphRAG assistant.
Answer the user's question ONLY using the provided RETRIEVED KNOWLEDGE GRAPH PATHWAYS and SOURCE TEXT EVIDENCE.
Explicitly trace the step-by-step entity relationships (e.g., Entity A -> [RELATION] -> Entity B) and cite the Chunk IDs.
If the graph context does not contain the answer, state that clearly.
"""

def ask_graphrag(question, G, k_hops=1, visualize=True):
    print("=" * 75)
    print(f"USER QUESTION: {question}")
    print("=" * 75)
    
    context_str, active_nodes, active_edges = retrieve_graph_context(question, G, k_hops=k_hops)
    print("\n" + context_str)
    
    final_prompt = f"{context_str}\n\nQUESTION: {question}\nANSWER:"
    answer = call_ollama(final_prompt, system_prompt=RAG_SYSTEM_PROMPT, json_mode=False)
    
    print("=" * 75)
    print("LLAMA 3.1 GRAPHRAG RESPONSE:")
    print("=" * 75)
    print(answer)
    
    if visualize and active_nodes:
        plot_knowledge_graph(
            G,
            highlight_nodes=active_nodes,
            highlight_edges=active_edges,
            title=f"Retrieved Multi-Hop Reasoning Subgraph for: '{question}'"
        )

# Run our multi-hop query!
query = "How is Demis Hassabis connected to Eli Lilly Oncology Targets?"
ask_graphrag(query, kg, k_hops=1, visualize=True)

## 6. Hands-On Exercises

* **Exercise 1 (Dynamic Incremental Ingestion):** Add a `Chunk 5` "Max Jaderberg leads the computational clinical platform for Eli Lilly Oncology Targets at Royal Marsden Research Center.
  Extract its triplets, merge them into `kg`, and recompute `nx.pagerank(kg)`.
  
* **Exercise 2 (All Simple Paths Explorer):** Write a function `explain_all_paths(G, source, target, cutoff=4)` using `nx.all_simple_paths(G.to_undirected(), source, target, cutoff=cutoff)` to find *every* distinct chain connecting `"Demis Hassabis"` and `"Isomorphic Labs"`.

  
* **Exercise 3 (Relation-Filtered Subgraph Retrieval):** Filter `kg` so that only edges with `relation in ["FUNDED", "DEVELOPS", "ADVANCES"]` are eligible during retrieval.